# 3장 2강: 프롬프트 설계와 활용 — 실습문제


## 실습 목표

- 프롬프트의 작업 목적·입력 데이터·판단 기준·출력 형식을 구분할 수 있다.
- 모호한 분류 요청을 구체적인 Zero-shot 프롬프트로 개선할 수 있다.
- 올바른 입력·출력 예시를 골라 Few-shot 프롬프트를 구성할 수 있다.
- 정보 추출과 요약의 목적에 맞는 출력 조건을 작성할 수 있다.
- 원문과 출력 조건을 기준으로 결과를 검토하고 필요한 조건을 보완할 수 있다.

## 실습 환경 / 데이터

- Python 3, Jupyter Notebook 또는 VS Code 노트북, pandas
- 아래 세 파일을 노트북과 같은 폴더에 둡니다. 다른 위치라면 `DATA_DIR`를 수정합니다.
- 앞 강의를 실행하지 않아도 됩니다. 이 노트북의 준비 코드부터 순서대로 실행하세요.

| 파일 | 이번 실습에서의 활용 |
|---|---|
| 엘지유플러스_분류.csv | 분류 대상 상담, 범주, Few-shot에 사용할 입력·출력 예시 확인 |
| 엘지유플러스_요약.csv | 요약 작업 지시, 포함할 핵심 내용, 분량 조건 확인 |
| 엘지유플러스_질의응답.csv | 상담에서 확인해야 할 사실과 질문·답변 확인 |

| 컬럼 | 의미와 사용 방법 |
|---|---|
| source_id | 상담 원문 ID. 같은 상담에서 여러 작업을 만들면 반복됨 |
| consulting_category | 데이터에 부여된 상담 분야 정보 |
| task_category | 상담 요건, 길이 제한 요약, 의문사형 등 세부 작업 유형 |
| instruction | CSV에 저장된 작업 지시 또는 질문 |
| input | 분석할 상담 원문 |
| output | CSV에 저장된 출력. 원문과 비교할 참고 자료 |

### 이번 실습의 범위

교안의 명언·리뷰 분석 사례를 실제 **상담 텍스트**에 적용합니다. 실습은 **프롬프트 설계와 검토**이며, 실제 LLM API 호출과 여러 문서의 자동 처리는 포함하지 않습니다.

- 실제 CSV 값, 사람이 작성한 기대 답안, 학습용 가상 출력은 구분하여 표시합니다.
- Python 코드는 원문 조회, 프롬프트 문자열 구성, 간단한 결과 표 작성에 사용합니다. 문자열을 출력하는 것은 모델 호출이 아닙니다.
- Zero-shot과 Few-shot의 차이는 **정답이 있는 입력·출력 예시를 함께 넣는지**입니다. 범주 정의나 출력 형식을 자세히 설명해도 정답 예시가 없으면 Zero-shot입니다.
- Few-shot용 예시는 원문과 저장된 출력을 확인한 뒤 사용합니다. 현재 분류할 대상의 정답은 프롬프트에 미리 넣지 않습니다.
- 저장된 `output`을 항상 옳다고 가정하지 않습니다. 분류 판단과 요약·추출 결과는 원문과 작업 기준으로 검토합니다.
- `<NAME>`, `<DATE>`, `<CHARGE>` 같은 원문의 가림 표시는 그대로 유지합니다. 가려진 사실을 추측하여 채우지 않습니다.
- 상담 속 상품 안내는 해당 상담을 해석하기 위한 자료이며, 현재 상품 정책을 확인한 자료가 아닙니다.

**구성:** 필수 1, 필수 2, 과제 1. 과제도 필수 문제와 같은 기초~중급 수준의 프롬프트 작성과 원문 검토를 수행합니다.


## 시작 전에 읽는 개념 정리

### 1. 프롬프트는 ‘무엇을, 어떤 기준으로, 어떤 형태로’ 요청할지 정하는 것입니다

“상담을 분석해 주세요”만으로는 분류가 필요한지, 요약이 필요한지, 특정 사실을 찾으라는 것인지 분명하지 않습니다. 프롬프트를 설계한다는 것은 **원하는 분석 작업과 조건을 문장으로 명확히 전달하는 일**입니다.

이 노트북은 프롬프트를 파이썬 문자열로 만들고 내용을 확인합니다. `print()`로 화면에 표시했다고 모델이 그 지시를 실행한 것은 아닙니다. 실제 LLM/API 호출은 포함하지 않으며, 기대 답안과 가상 출력도 구분해서 읽습니다.

| 구성 요소 | 쉬운 질문 | 이번 실습에서의 예 |
|---|---|---|
| 작업 목적 | 무엇을 하게 할까요? | 상담 요건 분류, 특정 사실 추출, 상담 요약 |
| 입력 데이터 | 무엇을 분석할까요? | 해당 상담의 `input` 전체 |
| 판단 기준 | 어떤 기준으로 내용을 해석할까요? | 중심 주제가 하나인지, 구분되는 주제가 둘 이상인지 |
| 출력 형식 | 결과를 어떻게 받겠습니까? | 범주명 하나, 항목명과 값, 두 문장 요약 |

판단 기준은 **답을 고르는 기준**, 출력 형식은 **고른 답을 적는 모양**입니다. 올바른 범주를 골라도 설명 문장을 길게 덧붙이면 ‘범주명 하나만’이라는 형식 조건에 맞지 않을 수 있습니다. 반대로 허용한 범주명만 적었어도 내용에 맞지 않는 범주일 수 있습니다.

### 2. 용어를 먼저 익히기

| 용어 | 쉬운 설명 | 혼동하지 않을 점 |
|---|---|---|
| 프롬프트 | 모델에 전달할 지시사항과 입력 정보 | 이번 코드에서는 전달할 문자열을 구성하는 단계까지 수행 |
| 작업 정의 | 수행할 분석을 구체적으로 정함 | ‘분석’이라는 넓은 말보다 분류·추출·요약처럼 명확히 지정 |
| 범주 | 분류 결과로 허용하는 항목 | 이 실습의 요건 분류는 단일·다수 두 범주 |
| 선택 규칙 | 가능한 항목 중 몇 개를 어떻게 고를지 정함 | 주제가 여러 개여도 출력은 ‘다수 요건 민원’이라는 범주 하나 |
| 출력 조건 | 포함할 내용·형태·분량에 관한 요구 | 길이 조건만으로 사실의 정확성을 보장하지 않음 |
| Zero-shot | 정답이 붙은 입력·출력 예시 없이 요청하는 방식 | 조건이 자세하거나 길어도 정답 예시가 없으면 해당 |
| Few-shot | 소수의 입력·정답 출력 쌍을 함께 제시하는 방식 | 입력 원문만 여러 개 붙이는 것과는 다름 |
| 기대 답안 | 원문과 기준을 보고 사람이 정한 비교 기준 | 이번 모델이 생성한 값이 아님 |
| 가상 출력 | 비교 연습을 위해 미리 정한 응답 예 | 실제 프롬프트 실행 결과가 아님 |
| 확인 불가 | 제공된 원문으로 값을 확정할 수 없다는 표시 | 값이 0이거나 해당 사실이 없다고 확정하는 뜻은 아님 |

### 3. 세 가지 분석 작업은 필요한 조건이 다릅니다

| 작업 | 결과로 받고 싶은 것 | 지시에 적어야 할 핵심 | 결과를 확인하는 방법 |
|---|---|---|---|
| 정보 추출 | 정해진 항목의 값 | 항목 이름, 원문 근거, 정보가 없을 때 처리 | 항목별 값·근거·누락 여부 확인 |
| 분류 | 정해진 범주 | 범주 정의, 선택 규칙, 범주명 출력 조건 | 내용에 맞는 범주인지와 형식 확인 |
| 요약 | 핵심 내용을 줄인 문장 | 포함할 정보, 분량, 사실 보존 조건 | 핵심 내용·사실·문장 수·길이 확인 |

동일한 상담을 활용하더라도 정보를 네 항목으로 추출하는 것과 두 문장으로 요약하는 것은 서로 다른 작업입니다. 따라서 모든 작업에 똑같은 출력 지시를 붙이기보다 목적에 맞는 조건을 씁니다.

### 4. 이번 강의에서 사용하는 상담의 역할

| 상담 ID | 이번 실습에서의 역할 | 사용 구간 |
|---:|---|---|
| 40005 | 단일·다수 요건으로 분류할 대상 | 필수 1과 필수 2에서 동일하게 사용 |
| 40001 | 판단 기준을 보여 줄 다른 상담 예시 | 필수 2의 Few-shot 예시 후보 |
| 40006 | 판단 기준을 보여 줄 다른 상담 예시 | 필수 2의 Few-shot 예시 후보 |
| 40003 | 필요한 사실과 핵심 요약을 설계할 대상 | 과제 1 |

예시의 정답은 Few-shot 프롬프트에 넣지만, **현재 분류할 대상의 정답은 미리 넣지 않습니다**. 대상의 기대 답안은 프롬프트와 별도 변수에 보관해 결과를 검토하는 기준으로 사용합니다.

프롬프트의 개선은 글자를 늘리는 일이 아닙니다. 먼저 어떤 문제가 있는지 확인하고, 그 문제와 관련된 조건을 보완합니다. 조건을 보완한 후에도 실제 응답이 좋아졌는지는 별도로 확인해야 합니다.


## 실습 준비

아래 **제공 코드**를 그대로 실행하세요. 파일 조회 함수 작성 자체는 평가하지 않습니다.

1. 세 파일의 크기·컬럼·자료형·결측치·빈 문자열을 확인합니다.
2. 고유 상담 수와 작업 유형 분포, 본문 길이를 확인합니다.
3. 상담 ID 40001·40003·40005·40006을 이번 실습 대상으로 확인합니다.
4. `get_record()`로 상담 ID와 작업 유형, 필요한 경우 질문까지 지정하여 한 행을 조회합니다.

`record_id`는 파일 이름과 0부터 시작하는 행 위치를 결합한 실습용 식별자입니다. 원본 파일의 컬럼이 아니며 파일 순서가 바뀌면 달라집니다. 같은 상담 ID에 여러 작업이 있으므로 `source_id`만으로 특정 작업 행을 고르지 않습니다.

본문 길이는 데이터 규모를 살펴보기 위한 **글자 수**입니다. LLM의 토큰 수나 API 사용량을 계산한 값은 아닙니다.


In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path(".")
classification = pd.read_csv(DATA_DIR / "엘지유플러스_분류.csv")
summarization = pd.read_csv(DATA_DIR / "엘지유플러스_요약.csv")
qa = pd.read_csv(DATA_DIR / "엘지유플러스_질의응답.csv")
tables = {"분류": classification, "요약": summarization, "질의응답": qa}

for name, table in tables.items():
    print(f"\n[{name}] 원본 {table.shape[0]:,}행 × {table.shape[1]}열")
    print(pd.DataFrame({"자료형": table.dtypes.astype(str),
                        "결측치": table.isna().sum()}).to_string())
    print("고유 상담 ID 수:", table["source_id"].nunique())
    print("빈 본문 수:", table["input"].str.strip().eq("").sum())
    print("빈 지시 수:", table["instruction"].str.strip().eq("").sum())
    print("빈 출력 수:", table["output"].str.strip().eq("").sum())
    print("작업 유형별 행 수:")
    print(table["task_category"].value_counts().to_string())
    print("행별 본문 글자 수(토큰 수가 아님):")
    print(table["input"].str.len().agg(["min", "mean", "max"]).round(2).to_string())
    table["record_id"] = [f"{name}-{i}" for i in range(len(table))]

practice_ids = [40001, 40003, 40005, 40006]
for source_id in practice_ids:
    texts = []
    print(f"\n[연습 상담 {source_id}]")
    for name, table in tables.items():
        part = table[table["source_id"] == source_id]
        print(name, "행 수:", len(part))
        texts.extend(part["input"].tolist())
    print("세 파일에서 이 상담의 원문이 모두 동일:", pd.Series(texts).nunique() == 1)
    print("이 상담 원문의 글자 수:", len(texts[0]))

def get_record(table, source_id, task_category, instruction=None):
    selected = table[(table["source_id"] == source_id)
                     & (table["task_category"] == task_category)]
    if instruction is not None:
        selected = selected[selected["instruction"] == instruction]
    if len(selected) != 1:
        raise ValueError("한 행을 선택해야 합니다. ID·작업 유형·질문을 다시 확인하세요.")
    return selected.iloc[0]

def show_record(row, include_input=False):
    print("\nrecord_id:", row["record_id"], " / source_id:", row["source_id"])
    print("작업 유형:", row["task_category"])
    print("지시 또는 질문:", row["instruction"])
    if include_input:
        print("상담 원문:\n" + row["input"])
    print("CSV에 저장된 출력:", row["output"])



[분류] 원본 16,080행 × 6열
                       자료형  결측치
source_id            int64    0
consulting_category    str    0
task_category          str    0
instruction            str    0
input                  str    0
output                 str    0
고유 상담 ID 수: 3216
빈 본문 수: 0
빈 지시 수: 0
빈 출력 수: 0
작업 유형별 행 수:
task_category
상담 요건    3216
상담 주제    3216
상담 사유    3216
상담 내용    3216
상담 결과    3216
행별 본문 글자 수(토큰 수가 아님):
min      670.00
mean    1583.42
max     2836.00

[요약] 원본 11,690행 × 6열
                       자료형  결측치
source_id            int64    0
consulting_category    str    0
task_category          str    0
instruction            str    0
input                  str    0
output                 str    0
고유 상담 ID 수: 3216
빈 본문 수: 0
빈 지시 수: 0
빈 출력 수: 0
작업 유형별 행 수:
task_category
기본 요약          3217
길이 제한 요약       2928
주제 중점 요약       2862
핵심 정보 추출 요약    2683
행별 본문 글자 수(토큰 수가 아님):
min      215.00
mean    1576.97
max     2836.00

[질의응답] 원본 7,497행 × 6열
                       자료형  결측치
source_id        

### 준비 코드 읽기 — 파일과 표의 구조부터 확인하기

`pandas`의 `DataFrame`은 행과 열로 이루어진 표입니다. 이번 데이터의 한 행은 ‘상담 한 개’ 자체보다 **특정 상담에 대한 작업 한 개**로 이해하면 좋습니다. 같은 상담에 분류·요약·질문을 여러 개 만들 수 있기 때문입니다.

| 코드·표현 | 하는 일 | 결과를 읽는 방법 |
|---|---|---|
| `Path(".")` | 현재 작업 폴더를 경로로 지정 | 데이터가 다른 폴더라면 해당 폴더로 수정 |
| `DATA_DIR / "엘지유플러스_분류.csv"` | 폴더와 파일 이름을 연결 | `/`는 여기서 경로 연결 역할 |
| `pd.read_csv(...)` | CSV를 표로 읽음 | 파일 조회이며 LLM 호출이 아님 |
| `tables = {"분류": classification, ...}` | 이름과 표를 짝지어 보관 | 세 파일에 같은 점검 절차를 적용하기 위함 |
| `tables.items()` | 이름과 표를 한 쌍씩 꺼냄 | 첫 반복에서 분류 표, 다음 반복에서 다른 표 확인 |
| `table.shape` | 표의 행 수·열 수 확인 | 행 수가 곧 고유 상담 수인 것은 아님 |
| `table.dtypes.astype(str)` | 열별 자료형 이름을 문자열로 표시 | 원본 상담 본문을 변환하는 작업이 아님 |
| `table.isna().sum()` | 열별 결측치 수 계산 | 비어 있는 값 확인 |
| `table["source_id"].nunique()` | 중복을 제외한 상담 ID 개수 계산 | 동일 ID의 작업 여러 행을 한 상담으로 셈 |
| `.str.strip().eq("").sum()` | 앞뒤 공백을 제외하면 빈 문자열인 값의 수 계산 | 결측치 검사와 별도로 내용 없는 문자열 확인 |
| `.value_counts()` | 작업 유형마다 몇 행인지 계산 | 상담 수보다 작업의 구성 확인에 초점 |
| `.str.len()` | 각 행의 본문 글자 수 계산 | 토큰 수를 계산하는 기능은 아님 |
| `.agg(["min", "mean", "max"])` | 길이의 최솟값·평균·최댓값 계산 | 파일에 있는 각 행을 기준으로 집계 |
| `.round(2)` | 계산값을 소수점 둘째 자리까지 반올림 | 본문을 자르거나 요약하는 기능이 아님 |

`print()`는 값을 화면에 보여 줍니다. `f"...{값}..."`는 문자열 중간에 값을 넣는 표기입니다. 예를 들어 `{table.shape[0]:,}`의 쉼표는 큰 행 수를 천 단위로 나누어 보여 줍니다. `.to_string(index=False)`는 표를 문자열로 출력하면서 왼쪽 인덱스를 숨기는 옵션이며, ID 열이나 실제 행을 삭제하지 않습니다.

### 같은 상담에 여러 작업이 있을 때의 ID

- `source_id`: 어느 상담에서 나온 자료인지 구분합니다.
- `task_category`: 그 상담에 대해 어떤 종류의 작업을 수행하는지 나타냅니다.
- `instruction`: 실제로 요구한 작업이나 질문 문장입니다.
- `record_id`: 현재 파일의 작업 행을 구별하기 위해 실습 코드가 추가한 이름입니다.

`[f"{name}-{i}" for i in range(len(table))]`는 현재 표의 각 행에 `분류-0`, `분류-1`처럼 이름을 붙입니다. `range(len(table))`는 0부터 행 수 직전까지의 번호를 만듭니다. 원본 표의 정렬 순서가 바뀌면 번호도 달라질 수 있으므로 원본 상담 ID와 같은 의미로 쓰지 않습니다.

준비 코드가 `record_id`를 추가한 뒤 표는 7열이 됩니다. 출력되는 ‘원본 6열’은 그 열을 추가하기 전에 확인한 값입니다. CSV 파일에 새 열을 저장하는 `to_csv()`는 실행하지 않습니다.


---
## 필수 1. 프롬프트 기본 구성과 Zero-shot 설계

### 문제 1-1. 모호한 분석 요청을 분류 프롬프트로 바꾸기

#### 문제 설명

상담 ID **40005**의 `상담 요건`을 분류하려고 합니다. 다음 요청은 구체적인 작업과 출력 조건이 부족합니다.

```text
다음 상담을 분석해 주세요.

상담 원문:
[40005의 input 전체]
```

사용할 범주는 CSV의 지시에 있는 `단일 요건 민원`, `다수 요건 민원`입니다. 다음은 이번 실습에 적용할 판단 기준입니다. 데이터 전체의 공식 라벨 정의를 새로 확정하는 것은 아닙니다.

- **단일 요건 민원:** 중심 요청·주제가 하나인 상담
- **다수 요건 민원:** 서로 구분되는 요청·주제가 두 가지 이상인 상담
- 같은 요청의 절차·준비물·처리 시점에 관한 세부 질문은 하나의 요건으로 봅니다.
- 다른 서비스 문제와 별도의 할인 상담이 함께 논의되면 각각의 주제를 확인합니다.
- 문장 수나 질문 횟수만으로 판단하지 않습니다.

#### 요구사항

1. `get_record(classification, 40005, "상담 요건")`으로 `target_row`를 선택하고 원문·기존 지시·저장된 출력을 확인하세요.
2. 위 모호한 요청에 원문 전체를 붙인 `weak_prompt`를 작성하세요. `weak_prompt = "다음 상담을 분석해 주세요.\n\n상담 원문:\n" + target_row["input"]`을 사용해도 됩니다.
3. 작업 목적, 입력 데이터, 판단 기준, 출력 형식을 모두 포함한 `zero_shot_prompt`를 작성하세요. 두 범주 중 **이름 하나만** 출력하도록 지정합니다. 정답 예시나 대상 상담의 저장된 `output`은 넣지 않습니다.
4. 네 구성 요소를 `구성 요소`, `프롬프트에 작성한 내용` 컬럼의 `prompt_components` 표로 정리하세요.
5. 원문을 읽고 기대 분류와 근거를 **프롬프트 밖에** 별도로 적으세요.

**보조 구문:** 여러 줄 문자열은 `"""..."""`로 작성할 수 있습니다. 문자열끼리 `+`로 연결하고 `print(프롬프트)`로 전체 내용을 확인합니다. 모델을 호출하는 코드는 작성하지 않습니다.

#### 해석 질문

**Q1.** 모호한 요청에는 무엇이 부족하며, 새 프롬프트의 네 구성 요소는 각각 무엇인가요?

**Q2.** 원문에 근거한 기대 분류는 무엇인가요? 질문 문장 수만 세거나, 실제로 완료된 업무만 세면 어떤 문제가 있나요?

**Q3.** 자세한 범주 정의와 출력 조건을 넣었는데도 Zero-shot인 이유는 무엇인가요? 대상 상담의 저장된 정답을 프롬프트에 넣지 않는 이유는 무엇인가요?

**Q4.** `다수 요건 민원`, `다수 요건으로 보입니다.`, `분류: 다수 요건 민원`이 나온다면 어떤 출력이 요구 형식에 맞나요? 프롬프트를 작성했다는 것만으로 실제 품질 개선을 확인한 것인가요?

#### 제출 결과

- 원문 조회 코드, `weak_prompt`, `zero_shot_prompt`
- `prompt_components` 표
- 기대 분류와 Q1~Q4 답변


### 풀이 전 안내 — 네 질문에 답하는 지시 작성하기

‘무엇을 할 것인가’, ‘무엇을 분석할 것인가’, ‘어떤 기준을 적용할 것인가’, ‘어떤 형태로 받을 것인가’를 먼저 짧게 적어 보세요. 이를 하나의 프롬프트로 연결하고 원문 전체가 실제 문자열에 포함되어 있는지 출력하여 확인합니다.

범주 이름만 적지 말고 두 범주의 차이와 세부 질문을 처리할 기준도 전달하세요. ‘다수’ 범주를 선택하는 것과 여러 범주명을 동시에 출력하는 것은 다릅니다. 문제의 출력 조건은 이름 하나입니다.

기대 분류와 근거는 프롬프트 밖에 작성하세요. 범주 정의와 출력 형식을 알려 주는 것은 가능하지만, 대상의 정답을 미리 제공하지 않는다는 조건을 지켜야 합니다.


In [2]:
# 필수 1 코드를 작성하세요.

# 1. 분류 대상 상담 40005의 상담 요건 행 확인
target_row = get_record(classification, 40005, "상담 요건")
show_record(target_row, include_input=True)

# 2. 모호한 요청 (작업, 기준, 출력 형식이 없음)
weak_prompt = "다음 상담을 분석해 주세요.\n\n상담 원문:\n" + target_row["input"]
print("\n[weak_prompt 앞부분]")
print(weak_prompt[:120] + " ...")
print("weak_prompt에 원문 전체 포함:", target_row["input"] in weak_prompt)

# 3. 네 구성 요소를 갖춘 Zero-shot 프롬프트 (정답 예시, 저장된 output은 넣지 않음)
task_instruction = "당신은 통신사 고객 상담 분류 담당자입니다. 아래 상담 원문 하나를 읽고 상담 요건을 분류하세요."

classification_rules = """[범주와 판단 기준]
- 단일 요건 민원: 상담의 중심 요청이나 주제가 하나인 경우
- 다수 요건 민원: 서로 구분되는 요청이나 주제가 두 가지 이상인 경우
- 같은 요청에 대한 절차, 준비물, 처리 시점을 묻는 세부 질문은 하나의 요건으로 봅니다.
- 서비스 문제와 별도의 할인 상담처럼 다른 주제가 함께 논의되면 각각을 별개의 주제로 확인합니다.
- 고객의 문장 수나 질문 횟수만으로 판단하지 않습니다.
- 실제로 처리가 완료되었는지와 관계없이 상담에서 논의된 주제를 기준으로 판단합니다."""

output_format = """[출력 형식]
- "단일 요건 민원", "다수 요건 민원" 중 범주 이름 하나만 그대로 출력하세요.
- 설명, 따옴표, "분류:" 같은 머리말, 문장부호를 붙이지 마세요."""

input_block = "[상담 원문]\n" + target_row["input"]

zero_shot_prompt = (task_instruction + "\n\n" + classification_rules + "\n\n"
                    + output_format + "\n\n" + input_block + "\n\n분류:")
print("\n[zero_shot_prompt 전체]")
print(zero_shot_prompt)
print("\nzero_shot_prompt에 원문 전체 포함:", target_row["input"] in zero_shot_prompt)

# 4. 네 구성 요소 정리
prompt_components = pd.DataFrame([
    {"구성 요소": "작업 목적", "프롬프트에 작성한 내용": task_instruction},
    {"구성 요소": "입력 데이터", "프롬프트에 작성한 내용": "상담 40005의 input 전체 (" + str(len(target_row["input"])) + "자)를 [상담 원문] 아래에 붙임"},
    {"구성 요소": "판단 기준", "프롬프트에 작성한 내용": "두 범주 정의, 세부 질문은 하나의 요건, 다른 주제가 함께 논의되면 각각 확인, 질문 횟수로 판단하지 않음"},
    {"구성 요소": "출력 형식", "프롬프트에 작성한 내용": "두 범주 중 이름 하나만, 설명과 머리말 없이 출력"},
])
print("\n[prompt_components]")
print(prompt_components.to_string(index=False))

# 5. 기대 분류와 근거 (프롬프트 밖에 따로 보관, 사람이 원문을 읽고 판단)
expected_class = "다수 요건 민원"
expected_reason = ("주제 1: 통화 중 대기 서비스가 제대로 안 되는 문제(삭제 후 재신청, 테스트 전화로 확인). "
                   "주제 2: 추가 결합 할인 혜택 안내와 추후 상담사 연락 요청. 서로 구분되는 주제가 두 가지임.")
print("\n[사람이 정한 기대 분류]", expected_class)
print("근거:", expected_reason)



record_id: 분류-23  / source_id: 40005
작업 유형: 상담 요건
지시 또는 질문: 이 상담의 내용이 주제가 하나인 "단일 요건 민원"인지, 여러 개의 주제가 있는 "다수 요건 민원"인지 알려 줘.
상담 원문:
상담사: 일상에 즐거운 변화 LGU+ <NAME>입니다.
고객: 네, 저 가입 그 뭐지 통화하고 있는 거 상대방이 알 수 있게 하는 거 있지 않습니까?
상담사: 어, 통화 중에 전화하면 대기가 걸리는 서비스 말씀하시는 거예요?
고객: 네.
상담사: 아, 네. 통화 중 대기라고 하는 서비스이고요.
고객: 네.
상담사: 그럼 신청을 하시는 우리 고객님 명의자분 성함이 <NAME> 고객님 본인 맞으실까요?
고객: <NAME>이요.
상담사: 아, 네, 본인 되십니까?
고객: 네.
상담사: 네, 반갑습니다. 잠시만 기다려 주세요.
고객: 네.
상담사: 이게 통화 중 대기 서비스가 원래 상대방한테 전화가 통화 중인 걸로 들어오면 통화 중입니다, 라고 해서 상대방한테 나오고 연결이 바로 되지가 않는데 통화 중 대기 부가 서비스를 신청을 하실 경우에는 상대방이 고객님께 전화를 걸면은 고객님한테는 뚜뚜 거리는 소리와 함께 걸려온 전화를 받을 수가 있고 상대방은 통화 중이라고 하는 멘트가 나오는 게 아니라 일반 신호음이 걸리는 거고요.
고객: 네.
상담사: 네, 근데 고객님은 지금 현재 봤을 때 통화 중 대기 부가 서비스 지금 현재 신청이 되어 있는 걸로 확인되시던데요.
고객: 아, 그래요? 근데 아까 전에 통화가 왔는데 녹취 들어가더라고요, 문자로.
상담사: 아, 문자로 그냥 매너콜 문자가 들어왔던가요, 고객님?
고객: 네.
상담사: 아, 그러면 고객님 제가 요거 통화 중 대기 서비스 삭제를 한 번 했고요.
고객: 네.
상담사: 다시 한번 신청해봐 드릴게요.
고객: 네.
상담사: 네, 고객님, 그럼 제가 테스트차 고객님한테 전화 한 번 드리겠습니다. 고객님 전화 받지 마시고요.
고객: 네.
상담사: 끊지 마시고 잠시만요.
고객: 네.
상담사: 

### 필수 1 답변 작성란

**Q1.**

- 답변: 모호한 요청은 "분석해 주세요"라고만 해서 분류를 하라는 건지 요약을 하라는 건지 작업 목적이 없고, 어떤 범주를 쓸지, 무엇을 기준으로 판단할지, 결과를 어떤 모양으로 줄지도 빠져 있다. 원문만 붙어 있는 상태다. 새 프롬프트의 네 구성 요소는 다음과 같다. 작업 목적은 "상담 원문 하나를 읽고 상담 요건을 분류"하는 것이다. 입력 데이터는 상담 40005의 input 전체(2,297자)로, [상담 원문] 아래에 붙였고 코드로 포함 여부를 확인했다(True). 판단 기준은 두 범주의 정의와 함께 같은 요청의 절차, 준비물, 처리 시점 질문은 하나의 요건으로 보고, 서비스 문제와 할인 상담처럼 다른 주제가 함께 나오면 각각 확인하고, 질문 횟수만으로 판단하지 않는다는 규칙이다. 출력 형식은 두 범주 중 이름 하나만, 설명이나 머리말 없이 쓰는 것이다.

**Q2.**

- 답변: 기대 분류는 "다수 요건 민원"이다. 상담 앞부분은 통화 중 대기 서비스가 제대로 안 되는 문제로, 상담사가 "통화 중 대기 서비스 삭제를 한 번 했고요", "다시 한번 신청해봐 드릴게요"라고 하고 테스트 전화로 확인했다. 뒷부분은 이와 별개로 상담사가 "결합 할인을 더 추가적으로 챙겨가실 수가 있는 할인 혜택"을 안내하고, 고객이 지금은 안 하겠다고 한 뒤 추후 연락에 동의한 결합 할인 상담이다. 서로 구분되는 주제가 두 개라서 다수 요건이다. 질문 문장 수만 세면 고객이 "네"라고만 하는 짧은 발화가 대부분이라 질문이 별로 없어서 단일로 잘못 볼 수 있고, 반대로 다른 상담에서는 같은 주제를 여러 번 물어도 다수로 잘못 세게 된다. 실제로 완료된 업무만 세면 할인은 가입하지 않았으니 통화 중 대기 처리 하나만 남아서 단일 요건으로 틀리게 판단하게 된다. 할인 상담도 상담에서 논의된 별도 주제이기 때문에 세야 한다.

**Q3.**

- 답변: Zero-shot과 Few-shot을 나누는 기준은 정답이 붙은 입력과 출력 예시를 같이 넣었느냐이다. 이번 프롬프트는 범주 정의와 판단 규칙, 출력 조건을 자세히 썼지만 "이 상담은 이 범주"라는 정답 예시는 하나도 없기 때문에 Zero-shot이다. 대상 상담의 저장된 정답을 프롬프트에 넣으면 모델이 판단하지 않고 답을 그대로 베껴 쓰게 되어서, 프롬프트가 제대로 작동하는지 확인할 수가 없다. 또 실제로 쓸 때는 새 상담의 정답을 미리 알 수 없으니 정답을 넣는 건 현실적이지도 않다. 그래서 기대 분류는 `expected_class`라는 별도 변수에 두고 결과를 검토하는 기준으로만 쓴다.

**Q4.**

- 답변: 요구 형식에 맞는 건 `다수 요건 민원` 하나뿐이다. `다수 요건으로 보입니다.`는 범주 이름이 아니라 설명 문장이고, `분류: 다수 요건 민원`은 내용은 맞지만 "분류:"라는 머리말이 붙어서 이름 하나만 쓰라는 조건에 어긋난다. 이런 값이 섞이면 저장하거나 집계할 때 같은 범주로 묶이지 않는다. 그리고 프롬프트를 작성했다는 것만으로 품질이 좋아졌다고 할 수는 없다. 이번 코드는 문자열을 만들어 `print()`로 보여 준 것이지 모델을 호출한 게 아니라서, 실제로 개선되었는지는 모델 응답을 받아 기대 답안과 같은 기준으로 비교해 봐야 알 수 있다.


---
## 필수 2. Few-shot 예시 선택과 비교

### 문제 2-1. 잘못된 예시를 고치고 Few-shot 구성하기

#### 문제 설명

필수 1과 같은 상담 **40005**를 같은 기준으로 분류합니다. 정답 예시 후보는 다음과 같습니다. 두 번째 후보에는 검토를 위한 오류를 넣었습니다.

| 예시 상담 ID | 가져올 본문 | 후보에 적힌 출력 |
|---:|---|---|
| 40001 | 분류 파일의 상담 요건 행 input | 다수 요건 민원 |
| 40006 | 분류 파일의 상담 요건 행 input | 다수 요건 민원 |

원문과 저장된 분류를 확인한 뒤 예시를 수정하세요. 40001은 여러 서비스 문의가 함께 있는 상담이고, 40006은 휴대폰 해지 방법·서류·처리 시점을 묻는 상담입니다.

또한 다음은 **같은 평가 대상 40005**에 대한 가상 비교 자료입니다. 실제 모델 실험 결과가 아니며, 실제로 이런 결과가 나왔다고 가정했을 때 어떻게 읽을지 연습합니다.

| 방식 | 가상 출력 |
|---|---|
| Zero-shot | 단일 요건 민원 |
| Few-shot | 다수 요건 민원 |

#### 요구사항

1. 40001·40006의 `상담 요건` 행을 각각 `example_multi`, `example_single`로 가져와 원문과 저장된 출력을 확인하세요.
2. `source_id`, `후보 출력`, `검토 후 출력`, `원문 근거`를 담은 `example_review` 표를 작성하세요. 잘못된 후보를 수정합니다.
3. 필수 1의 `classification_rules`를 유지하고, 검토한 두 상담의 **본문 전체와 올바른 분류**를 입력·출력 쌍으로 추가하여 `few_shot_prompt`를 작성하세요. 마지막에 40005의 본문과 빈 `분류:` 표시를 넣습니다. 대상의 정답은 넣지 않습니다.
4. 가상 비교 자료에 `허용 범주 여부`, `기대 분류와 일치`를 추가한 `comparison_table`을 작성하세요. 허용 범주는 두 개이며, 기대 분류는 필수 1에서 원문으로 판단한 값을 사용합니다.
5. 단순히 문자열이 일치하는지 확인한 결과와 실제 모델의 성능을 구분하여 해석하세요.

**보조 구문:** `문자열 + "\n" + 다른문자열`, `.isin([허용 범주])`, `표["가상 출력"] == expected_class`

#### 해석 질문

**Q1.** 어떤 예시 후보를 어떻게 수정해야 하나요? 해지 방법과 처리 시점을 따로 질문했다고 각각 별개의 요건으로 세면 왜 문제가 되나요?

**Q2.** 두 프롬프트의 공통 조건과 차이는 무엇인가요? 추가한 예시는 어떤 판단 기준을 보여 주나요?

**Q3.** 가상 비교 표에서 형식과 내용이 맞는 경우를 각각 설명하세요. 이 표를 실제 Few-shot 성능 향상의 근거라고 말할 수 있나요?

**Q4.** Zero-shot으로 충분한 결과를 얻는 경우와, 단일·다수 요건을 반복해서 혼동하는 경우에 각각 어떻게 접근하나요? Few-shot 예시를 고를 때 확인할 점을 세 가지 쓰세요.

#### 제출 결과

- 두 예시 상담의 원문 조회와 `example_review`
- `few_shot_prompt` 전체
- 가상 `comparison_table`과 Q1~Q4 답변


### 풀이 전 안내 — 예시는 붙이기 전에 검토하기

각 예시의 상담 요건 행을 조회하고 전체 원문을 읽으세요. 후보에 적힌 출력과 저장된 출력을 참고하되, 이번 기준으로 왜 맞고 틀린지 근거를 설명해야 합니다. 질문 개수만 세지 말고 같은 요청의 세부 절차인지 별개의 주제인지 나누어 보세요.

Few-shot에는 필수 1의 기준을 유지한 뒤, 검토한 예시의 본문과 정답을 짝지어 넣습니다. 마지막에는 현재 분류 대상의 본문과 답을 작성할 위치만 둡니다. 예시 정답과 대상 정답을 혼동하지 마세요.

가상 비교표의 `.isin()`은 허용 범주명 검사를, `== expected_class`는 준비한 기대 답안과의 문자열 비교를 수행합니다. 이 결과를 실제 모델 실행 결과로 설명하지 않습니다.


In [3]:
# 필수 2 코드를 작성하세요.

# 1. Few-shot 예시 후보 두 개의 원문과 저장된 출력 확인
example_multi = get_record(classification, 40001, "상담 요건")
example_single = get_record(classification, 40006, "상담 요건")
show_record(example_multi, include_input=True)
show_record(example_single, include_input=True)

# 2. 예시 후보 검토 (잘못된 후보 수정)
example_review = pd.DataFrame([
    {"source_id": 40001, "후보 출력": "다수 요건 민원", "검토 후 출력": "다수 요건 민원",
     "원문 근거": "'세 가지를 여쭤볼 거예요'라고 하고 케이스 배송, 삼성 케어 플러스 매장 코드, 카드 자동이체 할인, 아들 휴대폰 정지 횟수를 각각 물음"},
    {"source_id": 40006, "후보 출력": "다수 요건 민원", "검토 후 출력": "단일 요건 민원",
     "원문 근거": "해지 서류, 팩스, 대리점 방문, 처리 시간 질문이 모두 '휴대폰 해지'라는 하나의 요청에 대한 절차, 준비물, 처리 시점임"},
])
example_review["후보 수정 여부"] = example_review["후보 출력"] != example_review["검토 후 출력"]
print("\n[example_review]")
print(example_review.to_string(index=False))

# 3. Few-shot 프롬프트: 필수 1의 기준을 유지하고 입력과 정답 출력 쌍 두 개를 추가
few_shot_prompt = (
    task_instruction + "\n\n" + classification_rules + "\n\n" + output_format + "\n\n"
    + "[예시 1]\n상담 원문:\n" + example_multi["input"] + "\n분류: "
    + example_review.loc[0, "검토 후 출력"] + "\n\n"
    + "[예시 2]\n상담 원문:\n" + example_single["input"] + "\n분류: "
    + example_review.loc[1, "검토 후 출력"] + "\n\n"
    + "[분류할 상담]\n상담 원문:\n" + target_row["input"] + "\n분류:"
)
print("\n[few_shot_prompt 전체]")
print(few_shot_prompt)

# 프롬프트 구성 점검 (문자열 확인일 뿐 모델 실행이 아님)
print("\n분류 기준 포함:", classification_rules in few_shot_prompt)
print("예시 1 원문 포함:", example_multi["input"] in few_shot_prompt)
print("예시 2 원문 포함:", example_single["input"] in few_shot_prompt)
print("대상 원문 포함:", target_row["input"] in few_shot_prompt)
print("프롬프트가 빈 '분류:'로 끝남:", few_shot_prompt.endswith("분류:"))

# 4. 가상 비교 자료 (실제 모델 실행 결과가 아님)
allowed = ["단일 요건 민원", "다수 요건 민원"]
comparison_table = pd.DataFrame({
    "방식": ["Zero-shot", "Few-shot"],
    "가상 출력": ["단일 요건 민원", "다수 요건 민원"],
})
comparison_table["허용 범주 여부"] = comparison_table["가상 출력"].isin(allowed)
comparison_table["기대 분류와 일치"] = comparison_table["가상 출력"] == expected_class
print("\n[comparison_table] 가상 출력 기준, 기대 분류:", expected_class)
print(comparison_table.to_string(index=False))



record_id: 분류-0  / source_id: 40001
작업 유형: 상담 요건
지시 또는 질문: 상담 내용이 하나의 요건에 해당하는 "단일 요건 민원"과 여러 개의 요청이 들어있는 "다수 요건 민원" 중 어느 것에 해당하나요?
상담 원문:
상담사: 일상에 즐거운 변화 U+ <NAME>입니다.
고객: 네, 어, 제가 이제 플립3 사전예약 해서 그 케이스 신청했었는데 아직 안 오고 있거든요. 그거는 여기다 문의하는 거 아니죠?
상담사: 네, 어, 고객님, 죄송하지만 지금 상황 제가 여쭙고 케이스 관련해서 문의주신 거 확인해 드리려고 하는데 <NAME> 고객님 명의자분 본인 맞으십니까?
고객: 네, 세 가지를 여쭤볼 거예요. 한 가지가 그 케이스예요.
상담사: 아, 혹시 고객님 죄송하지만 그 대리점 쪽에서 하셨는지 아니면 직영점 쪽에서 하셨는지 아니면 저희 통화 요청하셔 가지고 저희 온라인 샵인 유샵 통해서 하셨는지요?
고객: 네, 음, 저기 직영점이요. 직영점인가, 거기가? 잘 모르겠네요.
상담사: 아, 직영점이요.
고객: 대리점은 아니에요. 아무튼 하나로 마트 옆에 있는 작은 샵이에요.
상담사: 아 그러세요? 아, 그러시면 그 가게에서 하신 거예요?
고객: 그렇죠, 예.
상담사: 아, 그러면 그 사은품 관련해서는 가게에서만 드리는 사은품이 있을 수 있어서 그거는 사은품 가게에도 한번 문의를 주셔야 될 거 같고 어, 다른 문의 사항 있으신 거는 말씀주시겠어요?
고객: 음, 네, 근데 이게 저기 케이스를 제가 온라인으로 주문을 했고요, 그거는. 네, 또 하나가 이제 사전 예약할 수 있는 삼성 케어 플러스라는 보험 상품이 있는데 지금 상품을 이제 제가 신청을 하는 거거든요. 신청을 했어요. 근데 매장 코드를 입력하라는 게 나왔는데, 매장 코드 입력하라는데 매장 코드가 뭔지 몰라서 저희가 이제 구입한 데 가서 매장 코드가 뭐냐고 물어봤더니 고객센터에 전화를 해보라는 거죠.
상담사: 아, 네.
고객: 자기들은 이 매장 코드는 모른다는 거예요.
상담사: 고객님,

### 필수 2 답변 작성란

**Q1.**

- 답변: 40001 후보 "다수 요건 민원"은 그대로 두면 된다. 고객이 직접 "세 가지를 여쭤볼 거예요"라고 했고 실제로 케이스 배송, 삼성 케어 플러스 매장 코드, 카드 자동이체 할인, 아들 휴대폰 정지 횟수처럼 서로 다른 주제를 물었다. 40006 후보 "다수 요건 민원"은 "단일 요건 민원"으로 고쳐야 한다(저장된 출력도 단일 요건 민원이었다). 40006은 해지 서류, 팩스 가능 여부, 대리점 방문, 해지 처리 시간 등을 물었지만 모두 "휴대폰 해지"라는 요청 하나에 대한 절차, 준비물, 처리 시점이다. 이걸 질문마다 별개 요건으로 세면 질문 횟수만으로 판단하는 것과 같아져서, 하나의 일을 자세히 물어본 상담이 전부 다수 요건이 되어 버린다. 더구나 잘못된 예시가 Few-shot에 들어가면 모델이 그 잘못된 기준을 따라 배우게 되니까 더 문제가 된다.

**Q2.**

- 답변: 두 프롬프트의 공통 조건은 작업 목적(`task_instruction`), 판단 기준(`classification_rules`), 출력 형식(`output_format`), 그리고 대상 40005의 원문 전체이다. 차이는 Few-shot에만 40001과 40006의 원문 전체와 올바른 분류가 입력과 출력 쌍으로 두 개 들어 있다는 점이다. 마지막은 둘 다 대상 원문 뒤에 빈 "분류:"만 두었고 대상의 정답은 넣지 않았다. 40001 예시는 서로 다른 서비스 주제가 여러 개 나오면 다수 요건이라는 기준을 보여 주고, 40006 예시는 질문이 많아도 같은 요청의 세부 절차라면 단일 요건이라는 기준을 보여 준다. 두 예시가 반대 범주라서 질문 수가 아니라 주제 수로 판단한다는 점을 같이 보여 줄 수 있다.

**Q3.**

- 답변: 형식만 보면 Zero-shot의 "단일 요건 민원"과 Few-shot의 "다수 요건 민원" 둘 다 허용 범주(True)라서 형식은 둘 다 맞다. 내용은 기대 분류 "다수 요건 민원"과 비교했을 때 Few-shot만 일치(True)하고 Zero-shot은 불일치(False)이다. 하지만 이 표는 미리 정한 가상 출력을 문자열로 비교한 것일 뿐이고, 상담 한 건에 대한 한 번의 결과라서 실제 Few-shot 성능이 좋아졌다는 근거라고 말할 수 없다. 실제로 말하려면 모델을 직접 호출해서 여러 상담에 대해 같은 기준으로 결과를 모아 비교해야 한다.

**Q4.**

- 답변: Zero-shot으로도 원하는 범주를 형식에 맞게 잘 내면 그대로 쓰는 게 낫다. 예시를 넣으면 프롬프트가 길어지고 예시 고르는 수고도 들기 때문이다. 반대로 단일과 다수 요건을 계속 헷갈린다면, 먼저 판단 기준 문장이 분명한지 보완하고, 그래도 안 되면 헷갈리는 유형을 보여 주는 Few-shot 예시를 넣는다. Few-shot 예시를 고를 때 확인할 점은 첫째, 예시의 정답이 원문과 기준에 비추어 정말 맞는지(40006처럼 후보가 틀릴 수 있음), 둘째, 단일과 다수처럼 범주가 한쪽으로 쏠리지 않고 서로 비교되는 예시를 골랐는지, 셋째, 분류할 대상 상담 자체나 대상의 정답이 예시에 섞여 들어가지 않았는지이다.


---
## 과제 1. 정보 추출·요약 목적에 맞는 프롬프트 설계

### 문제 3-1. 같은 상담에서 필요한 사실과 요약 조건 정하기

#### 문제 설명

상담 ID **40003**을 대상으로, 필수 문제에서 익힌 네 구성 요소를 정보 추출과 요약에 적용합니다. 다음 행을 확인하세요.

| 변수명 | 파일 | task_category | instruction 선택 조건 |
|---|---|---|---|
| owner_row | qa | 의문사형 | 고객의 결합 할인은 누구 앞으로 되어 있나요? |
| keep_row | qa | 예/아니요형 | 고객의 요금제 변경 시 기존 결합 할인은 유지되나요? |
| short_summary_row | summarization | 길이 제한 요약 | 추가 조건 없음 |

정보 추출 항목은 **변경한 요금제, 기존 결합 할인 유지 여부, 결합 할인 지정 대상, 변경 후 정확한 월 요금(원)**입니다. 원문에서 확인되지 않는 값은 `확인 불가`로 적도록 설계합니다.

요약은 **2문장, 300자 이내**로 작성하도록 설계하고, **요금제 변경 처리, 기존 결합 할인 유지 여부, 멤버십 등급 변경**을 반드시 포함하도록 지정합니다. 원문에 없는 사실을 추가하거나 의미를 바꾸지 않아야 합니다.

다음은 프롬프트의 보완점을 찾기 위한 **가상 오류 결과**입니다. 실제 모델을 실행한 결과가 아닙니다.

```text
[정보 추출의 일부]
결합 할인 지정 대상: 고객 본인
변경 후 정확한 월 요금(원): 37500

[요약]
고객은 5G 슬림+로 변경했고 기존 결합 할인은 해지되었다.
```

#### 요구사항

1. 지정한 세 행을 조회하고, 상담 원문은 한 번, 지시와 저장된 출력은 각각 출력하세요. 세 `input`이 같은지도 확인합니다.
2. 네 추출 항목, 원문 근거 조건, `확인 불가` 처리 기준, `항목명: 값` 출력 형식을 갖춘 `extraction_prompt`를 작성하세요. 원문 전체를 붙입니다.
3. 포함할 세 가지 핵심 정보, 분량, 사실 보존 조건을 갖춘 `summary_prompt`를 작성하세요. 원문 전체를 붙입니다. 두 프롬프트 모두 정답 예시를 넣지 않습니다.
4. 원문을 읽고 정보 추출의 기대 답안과 2문장 요약을 **프롬프트 밖에** 직접 작성하세요. 요약 길이는 `len(요약문)`으로 확인할 수 있습니다.
5. 가상 오류 결과를 `문제 부분`, `원문에 따른 수정`, `보완할 프롬프트 조건` 컬럼의 `improvement_table`로 검토하세요. 정보 오류뿐 아니라 요약의 핵심 정보 누락과 분량 조건도 확인합니다.

#### 해석 질문

**Q1.** 정보 추출과 요약의 목적·출력 형태·주요 조건은 어떻게 다른가요? 이번 과제의 두 프롬프트는 Zero-shot인가요, Few-shot인가요?

**Q2.** 추출 항목 네 개의 기대 값과 각각의 근거를 설명하세요. 원문에 `5G 다이렉트 37.5`라는 표현이 있다고 변경한 요금제의 정확한 월 요금을 37,500원으로 채워도 되나요?

**Q3.** 가상 오류 결과에서 무엇을 수정하고 프롬프트의 어떤 조건을 보완해야 하나요? 2문장·300자 이내라는 형식만 맞으면 좋은 요약인가요?

**Q4.** 프롬프트 수정 전후를 어떤 데이터와 기준으로 비교해야 하나요? 원문에 금액이 없으면 예시를 추가하거나 지시를 길게 쓰는 것만으로 정확한 금액을 만들 수 있나요?

#### 제출 결과

- 세 행 조회와 원문 동일성 확인
- `extraction_prompt`, `summary_prompt`
- 사람이 작성한 추출 기대 답안과 2문장 요약
- `improvement_table`과 Q1~Q4 답변


### 풀이 전 안내 — 작업마다 조건을 다르게 구성하기

정보 추출에는 네 항목과 각 항목의 근거 조건, 확인할 수 없을 때의 처리 방법을 적으세요. 요약에는 반드시 담을 세 핵심 내용과 두 문장·300자 이내 조건을 적습니다. 두 프롬프트에 같은 원문을 사용하더라도 결과 형태는 다르게 요청합니다.

원문에 여러 요금제가 나올 때는 문의만 한 것과 실제로 변경한 것을 구분하세요. 가려진 금액이나 날짜는 주변의 다른 숫자로 채우지 않습니다. 기대 답안은 원문을 읽어 프롬프트 밖에 작성합니다.

오류 검토표에서는 ‘어떤 부분이 잘못되었는지’, ‘원문 기준으로 어떻게 고칠지’, ‘그 문제를 줄이려면 어떤 지시를 명확히 할지’를 한 행에 연결하세요. 요약은 내용과 형식 모두 확인하며, `len()`은 글자 수만 계산한다는 점에 유의합니다.


In [4]:
# 과제 1 코드를 작성하세요.

# 1. 상담 40003의 세 행 조회
owner_row = get_record(qa, 40003, "의문사형", "고객의 결합 할인은 누구 앞으로 되어 있나요?")
keep_row = get_record(qa, 40003, "예/아니요형", "고객의 요금제 변경 시 기존 결합 할인은 유지되나요?")
short_summary_row = get_record(summarization, 40003, "길이 제한 요약")

show_record(owner_row, include_input=True)
show_record(keep_row)
show_record(short_summary_row)
print("\n세 행의 input 동일 여부:",
      owner_row["input"] == keep_row["input"] == short_summary_row["input"])

source_text = owner_row["input"]

# 2. 정보 추출 프롬프트 (정답 예시 없음)
extraction_prompt = """당신은 통신사 상담 기록에서 정보를 추출하는 담당자입니다.
아래 상담 원문을 읽고 다음 네 항목의 값을 추출하세요.

[추출 항목]
1. 변경한 요금제: 상담 중 실제로 변경 처리된 요금제 이름 (문의만 하고 변경하지 않은 요금제는 제외)
2. 기존 결합 할인 유지 여부: 요금제 변경 후 기존 결합 할인이 유지되는지 (유지됨 / 유지되지 않음)
3. 결합 할인 지정 대상: 결합 할인이 누구 앞으로 지정되어 있는지
4. 변경 후 정확한 월 요금(원): 변경한 요금제의 정확한 월 요금 숫자

[작성 기준]
- 모든 값은 상담 원문에서 직접 확인되는 내용만 사용하세요.
- 원문에 값이 없거나 <NAME>, <DATE>, <CHARGE>처럼 가려져 있으면 추측하지 말고 "확인 불가"라고 쓰세요.
- 요금제 이름에 들어 있는 숫자나 다른 요금제의 숫자를 월 요금으로 바꾸어 쓰지 마세요.

[출력 형식]
항목명: 값 형식으로 네 줄만 출력하고 다른 설명은 붙이지 마세요.
변경한 요금제: 
기존 결합 할인 유지 여부: 
결합 할인 지정 대상: 
변경 후 정확한 월 요금(원): 

[상담 원문]
""" + source_text

# 3. 요약 프롬프트 (정답 예시 없음)
summary_prompt = """당신은 통신사 상담 내용을 요약하는 담당자입니다.
아래 상담 원문을 읽고 요약하세요.

[반드시 포함할 내용]
1. 요금제 변경 처리: 어떤 요금제로 실제 변경했는지
2. 기존 결합 할인 유지 여부: 요금제 변경 후에도 결합 할인이 유지되는지
3. 멤버십 등급 변경: 요금제 변경으로 멤버십 등급이 어떻게 바뀌었는지

[분량]
- 정확히 2문장, 공백 포함 300자 이내로 작성하세요.

[사실 보존 조건]
- 원문에 있는 사실만 쓰고, 원문에 없는 내용(보상, 정확한 금액 등)을 추가하지 마세요.
- 유지, 해지, 변경 같은 상태를 원문과 다르게 바꾸지 마세요.
- <NAME>, <DATE>, <CHARGE> 같은 가림 표시는 추측해서 채우지 마세요.

[상담 원문]
""" + source_text

print("\n[extraction_prompt 전체]")
print(extraction_prompt)
print("\n[summary_prompt 전체]")
print(summary_prompt)
print("\n두 프롬프트에 원문 전체 포함:", source_text in extraction_prompt, source_text in summary_prompt)

# 4. 사람이 원문을 읽고 작성한 기대 답안 (프롬프트 밖)
expected_extraction = pd.DataFrame([
    {"항목": "변경한 요금제", "기대 값": "5G 슬림+",
     "원문 근거": "'5G 슬림+ 그 <CHARGE> 짜리는 가능해요?' '네, 가능합니다.' 후 '지금 즉시 변경해 놓고'"},
    {"항목": "기존 결합 할인 유지 여부", "기대 값": "유지됨",
     "원문 근거": "'그 할인은 그냥 기존대로 들어간다는 거죠?' '네, 맞습니다.'"},
    {"항목": "결합 할인 지정 대상", "기대 값": "남편",
     "원문 근거": "'아, 저희 남편 앞으로 되어 있나요?' '어, 네, 맞습니다.'"},
    {"항목": "변경 후 정확한 월 요금(원)", "기대 값": "확인 불가",
     "원문 근거": "5G 슬림+ 요금이 '<CHARGE>'로 가려져 있음. 37.5는 변경하지 않은 다이렉트 요금제 이름"},
])
print("\n[사람이 작성한 정보 추출 기대 답안]")
print(expected_extraction.to_string(index=False))

my_summary = ("고객은 위약금 없이 변경 가능한 날짜를 확인한 뒤 요청하여 요금제를 5G 슬림+로 즉시 변경했고, "
              "남편 앞으로 지정된 기존 결합 할인은 요금제를 바꿔도 그대로 유지된다고 안내받았다. "
              "다만 요금제가 낮아져 멤버십 등급이 VIP에서 일반 플러스로 바뀌어 VIP 혜택은 받을 수 없게 되었다.")
print("\n[사람이 작성한 2문장 요약]")
print(my_summary)
print("문장 수:", my_summary.count("다."), "/ 글자 수:", len(my_summary), "/ 300자 이내:", len(my_summary) <= 300)

# 원문 확인: 37.5와 가림 표시
print("\n원문에 '37.5' 포함:", "37.5" in source_text, "/ 원문 속 <CHARGE> 개수:", source_text.count("<CHARGE>"))

# 5. 가상 오류 결과 검토 (실제 모델 결과가 아님)
fake_summary = "고객은 5G 슬림+로 변경했고 기존 결합 할인은 해지되었다."
print("가상 요약 문장 수:", fake_summary.count("다."), "/ 글자 수:", len(fake_summary),
      "/ '멤버십' 포함:", "멤버십" in fake_summary)

improvement_table = pd.DataFrame([
    {"문제 부분": "결합 할인 지정 대상: 고객 본인",
     "원문에 따른 수정": "남편",
     "보완할 프롬프트 조건": "지정 대상은 상담사가 확인해 준 발화를 근거로 찾고, 상담 중인 고객을 기본값으로 쓰지 말 것"},
    {"문제 부분": "변경 후 정확한 월 요금(원): 37500",
     "원문에 따른 수정": "확인 불가",
     "보완할 프롬프트 조건": "가려진 <CHARGE>나 다른 요금제 이름 속 숫자로 금액을 채우지 말고 확인 불가로 쓸 것"},
    {"문제 부분": "요약: 기존 결합 할인은 해지되었다",
     "원문에 따른 수정": "기존 결합 할인은 그대로 유지된다",
     "보완할 프롬프트 조건": "유지, 해지 같은 상태는 원문의 답변 그대로 쓰고 바꾸지 말 것"},
    {"문제 부분": "요약: 멤버십 등급 변경 누락",
     "원문에 따른 수정": "멤버십 등급이 VIP에서 일반 플러스로 바뀐 내용 추가",
     "보완할 프롬프트 조건": "세 가지 필수 내용을 모두 포함했는지 출력 전에 확인하도록 지시"},
    {"문제 부분": "요약: 1문장만 작성",
     "원문에 따른 수정": "2문장으로 작성",
     "보완할 프롬프트 조건": "글자 수뿐 아니라 '정확히 2문장' 조건을 분명히 쓸 것"},
])
print("\n[improvement_table]")
print(improvement_table.to_string(index=False))



record_id: 질의응답-6  / source_id: 40003
작업 유형: 의문사형
지시 또는 질문: 고객의 결합 할인은 누구 앞으로 되어 있나요?
상담 원문:
상담사: 일상에 즐거운 변화 LGU+ <NAME>입니다.
고객: 아, 네, 안녕하세요? 저<DATE> 날 휴대폰을 바꿨는데요.
상담사: 네.
고객: 어, 제가 요금제를 그때 대리점에서는 <DATE> 요금제를 바꿀 수 있다고 했는데 제가 예전에 <PHONE_NUMBER> 통화했을 때는 <DATE> 바꿀 수 있다고 했거든요.
상담사: 아, 네.
고객: 네, 한번 확인을 좀 해주실 수 있을까요?
상담사: 네, 확인해 보겠습니다. 입력해 주신 번호가 <NAME> 고객님 본인 맞으십니까?
고객: 아, 네, 맞아요.
상담사: 감사합니다. <DATE>부터 변경 가능하신 날짜가 맞네요. 변경하셔도 되고요.
고객: 어, 네. 근데 그게 저한테 그 대리점에서 그 전에 바꾸면 위약금이 있다는 식으로 얘기했거든요, <DATE> 전에. 네.
상담사: 아, 네, <DATE>부터, 네.
고객: 가능한 거예요?
상담사: <DATE>부터 변경하셔도 위약금 안 나오는 기간 맞습니다.
고객: 아, 네.
상담사: 매장 쪽에서 이제 아무래도 기간 자체를 조금 넉넉하게 기준 잡다 보니까 하루 이틀 정도 뒤로 말씀을 드린 거 같은데요.
고객: 아, 네.
상담사: 네, <DATE>부터 변경하실 수 있는 게 맞아요.
고객: 아, 네, 그럼 혹시 5G 다이렉트 37.5라는 그 요금제도 제가 할 수 있는 건가요?
상담사: 어, 다이렉트 요금제인데요. 고객님, 근데 이게 조건이 휴대폰 기기를 구매하실 때 저희 U+샵 부서를 통해서 개통을 해주셔야 되고요. 그리고 일부 이런 결합 할인이나 약정이 저희가 들어가시면 안 되는 요금제이기 때문에, 네.
고객: 네, 아... 그러면 안 되네요.
상담사: 네, 맞습니다.
고객: 아, 그럼 혹시 5G 슬림+ 그 <CHARGE> 짜리는 가능해요?
상담사: 네, 가능합니다.
고객: 아, 그러면 그걸로 만약에 <

### 과제 1 답변 작성란

**Q1.**

- 답변: 정보 추출은 정해진 항목의 값을 원문에서 정확히 찾아내는 게 목적이고, 출력은 "항목명: 값" 형태의 네 줄이다. 주요 조건은 항목 이름, 원문 근거로만 채우기, 값이 없거나 가려져 있으면 "확인 불가"로 쓰기이다. 요약은 상담의 핵심 흐름을 짧게 줄이는 게 목적이고, 출력은 2문장짜리 글이다. 주요 조건은 꼭 들어가야 할 세 가지 내용(요금제 변경 처리, 결합 할인 유지 여부, 멤버십 등급 변경), 2문장, 300자 이내라는 분량, 원문 사실을 바꾸거나 추가하지 않는 것이다. 두 프롬프트 모두 정답이 붙은 예시 없이 조건만 적었기 때문에 Zero-shot이다.

**Q2.**

- 답변: 변경한 요금제는 "5G 슬림+"이다. 고객이 "5G 슬림+ 그 <CHARGE> 짜리는 가능해요?"라고 묻자 "네, 가능합니다"라고 했고, 마지막에 상담사가 "지금 즉시 변경해 놓고"라고 했다. 기존 결합 할인 유지 여부는 "유지됨"이다. "그 할인은 그냥 기존대로 들어간다는 거죠?"에 "네, 맞습니다"라고 답했다. 결합 할인 지정 대상은 "남편"이다. "저희 남편 앞으로 되어 있나요?"에 "어, 네, 맞습니다"라고 했다. 변경 후 정확한 월 요금은 "확인 불가"이다. 5G 슬림+의 요금은 원문에서 `<CHARGE>`로 가려져 있다(원문에 <CHARGE>가 6번 나옴). `5G 다이렉트 37.5`라는 표현이 원문에 있긴 하지만, 그건 고객이 문의만 하고 U+샵 개통 조건 때문에 "그러면 안 되네요"라고 포기한 다른 요금제의 이름이다. 그래서 변경한 요금제의 월 요금을 37,500원으로 채우면 안 된다.

**Q3.**

- 답변: 정보 추출에서 "결합 할인 지정 대상: 고객 본인"은 "남편"으로, "변경 후 정확한 월 요금(원): 37500"은 "확인 불가"로 고쳐야 한다. 요약에서 "기존 결합 할인은 해지되었다"는 "그대로 유지된다"로 고쳐야 하고, 멤버십 등급이 VIP에서 일반 플러스로 바뀐 내용이 빠져 있어서 추가해야 한다. 또 가상 요약은 1문장(33자)이라 2문장 조건도 못 지켰다. 그래서 프롬프트에는 지정 대상은 상담사가 확인해 준 발화로 찾을 것, 가려진 값이나 다른 요금제 숫자로 금액을 채우지 말 것, 유지나 해지 같은 상태를 원문대로 쓸 것, 세 가지 필수 내용을 다 넣었는지 확인할 것, 정확히 2문장으로 쓸 것을 분명히 해야 한다. 그리고 2문장, 300자 이내라는 형식만 맞는다고 좋은 요약이 아니다. 내가 쓴 요약은 160자, 2문장이라 형식도 맞지만, 그보다 결합 할인처럼 사실이 원문과 맞는지와 필수 내용이 빠지지 않았는지가 더 중요하다. `len()`은 글자 수만 세 주기 때문에 내용은 사람이 따로 확인해야 한다.

**Q4.**

- 답변: 수정 전과 후를 비교할 때는 같은 상담 데이터(이번에는 40003, 가능하면 다른 상담도 여러 건)에 두 프롬프트를 똑같이 적용하고, 미리 원문을 읽고 만든 기대 답안을 기준으로 비교해야 한다. 기준은 항목별 값이 기대 답안과 맞는지, 확인 불가를 제대로 썼는지, 요약에 필수 세 내용이 다 들어 있는지, 원문에 없는 내용이 추가되지 않았는지, 2문장과 300자 이내 조건을 지켰는지이다. 그리고 원문에 금액이 `<CHARGE>`로 가려져 있으면 예시를 추가하거나 지시를 길게 쓴다고 정확한 금액이 생기지 않는다. 프롬프트는 원문에 있는 정보를 잘 꺼내게 도와줄 뿐이라서, 없는 값을 요구하면 오히려 37500 같은 그럴듯한 숫자를 지어낼 위험만 커진다. 이런 경우에는 확인 불가로 두는 게 맞다.


## 개념 정리 — 이번 강의에서 구분할 것

| 구분 | 첫 번째 의미 | 두 번째 의미 |
|---|---|---|
| 작업 목적 / 입력 데이터 | 무엇을 수행할지 | 무엇을 분석할지 |
| 판단 기준 / 출력 형식 | 어떤 의미 기준으로 답할지 | 결과를 어떤 모양으로 적을지 |
| 범주 정의 / 정답 예시 | 범주가 뜻하는 내용 | 특정 입력을 어떤 범주로 판단했는지 보여 주는 사례 |
| Zero-shot / Few-shot | 정답 예시 없이 요청 | 입력·정답 출력 쌍을 함께 제시 |
| 예시 상담 / 분류 대상 | 판단 사례를 보여 주는 자료 | 지금 답을 얻고 싶은 자료 |
| 프롬프트 문자열 / 모델 응답 | 모델에 전달할 요청 내용 | 실제 호출 뒤 생성된 결과 |
| 기대 답안 / 가상 출력 | 원문을 보고 사람이 정한 비교 기준 | 비교 연습을 위해 미리 정한 결과 |
| 허용 범주 여부 / 기대 분류 일치 | 허용한 범주명인지 검사 | 준비한 기대 범주와 같은지 검사 |
| 확인 불가 / 값이 0 | 제공된 자료로 확정할 수 없음 | 숫자가 0이라는 구체적인 값 |
| 조건 보완 / 품질 개선 확인 | 필요한 지시를 더 명확히 작성 | 실제 응답을 같은 기준으로 다시 비교 |

설명은 ‘어떤 작업을 요청했는가 → 어떤 기준과 형식을 정했는가 → 원문에 어떤 근거가 있는가 → 결과를 무엇으로 확인할 것인가’ 순서로 작성해 보세요. 프롬프트를 길게 썼다는 사실보다 각 문구가 어떤 모호함이나 오류를 줄이려는지 설명하는 것이 중요합니다.


---
## 실습 마무리

다음 질문에 자신의 말로 답하세요.

1. 이번 실습에서 프롬프트로 해결하려던 분석 문제는 무엇인가요?
2. 네 구성 요소를 어떻게 명확히 했나요?
3. Zero-shot과 Few-shot을 구분하는 기준과 선택 기준은 무엇인가요?
4. 정보 추출·분류·요약 결과는 각각 무엇을 근거로 검증하나요?
5. 이번에 실제 실행한 작업과 아직 확인하지 않은 결과는 무엇인가요?


### 실습 마무리 답변 작성란

1. 상담 원문을 보고 상담 요건 분류(단일, 다수), 특정 사실 정보 추출, 핵심 요약을 정확하게 하도록 요청하는 프롬프트를 만드는 것이었다. 특히 "분석해 주세요"처럼 모호한 요청 때문에 생기는 결과의 차이와 오류를 줄이는 게 목표였다.

2. 작업 목적은 "상담 요건을 분류하세요"처럼 할 일을 구체적으로 적었고, 입력 데이터는 해당 상담의 input 전체를 [상담 원문] 아래에 붙였다. 판단 기준은 범주 정의와 세부 질문 처리 규칙, 확인 불가 처리 기준 같은 것으로 적었고, 출력 형식은 범주 이름 하나, "항목명: 값" 네 줄, 2문장 300자 이내처럼 작업마다 다르게 정했다.

3. 구분 기준은 정답이 붙은 입력과 출력 예시를 넣었는지이다. 조건이 아무리 길어도 예시가 없으면 Zero-shot이다. 조건만으로 잘 되는 작업은 Zero-shot을 쓰고, 단일과 다수 요건처럼 헷갈리는 판단이 반복되면 원문으로 검토한 올바른 예시를 골라 Few-shot을 쓴다.

4. 정보 추출은 항목별 값이 원문의 어느 발화로 확인되는지, 가려진 값을 확인 불가로 처리했는지로 검증한다. 분류는 허용 범주인지와 함께 원문의 주제 수가 범주 정의에 맞는지로 검증한다. 요약은 필수 내용 포함 여부, 원문 사실과 일치하는지, 문장 수와 글자 수 조건을 지켰는지로 검증한다.

5. 이번에 실제로 한 건 CSV 조회, 프롬프트 문자열 작성과 출력, 원문을 읽고 기대 답안 작성, 가상 출력과 기대 답안의 문자열 비교, 글자 수 확인이다. 아직 확인하지 않은 건 이 프롬프트를 실제 LLM에 넣었을 때 나오는 응답과, 그 응답이 Zero-shot보다 Few-shot에서 정말 좋아지는지, 그리고 다른 상담들에서도 잘 되는지이다.
